In [9]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt
from utils import is_casenum


In [10]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 8000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 4000
REQUESTS_PER_BATCH = 1000

In [11]:
MIN_IDX = 0
MAX_IDX = 0
REPLACE = False
REPLACE_OUTPUT_FILE = True
MODE = 'WRITE'  # ESTIMATE | BATCH | WRITE

In [12]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [13]:
PROMPT = Template("""
In what follows, I will provide you with an agenda item from a Los Angeles City Planning Commission meeting, and the full minutes text for that meeting. 

Read the agenda item and minutes carefully and extract the relevant information as specified below.

Return a JSON in the following format:

{
    "summary": "<One paragraph summary of the deliberations related to the agenda item and the final decision made by the Commission.>",
    "item_order_changed": "<Whether the agenda item order was changed during the meeting; your options are: YES | NO>",
    "taken_off_consent_calendar": "<Whether the agenda item was taken off the consent calendar; your options are: YES | NO>",
    "multiple_votes": "<Whether there were multiple votes on the agenda item; your options are: YES | NO>",
    "mover": "<The name of the member who made the motion.>",
    "seconder": "<The name of the member who seconded the motion.>",
    "ayes": ["<List of names of members who voted in favor of the motion.>"],
    "noes": ["<List of names of members who voted against the motion.>"],
    "abstentions": ["<List of names of members who abstained from voting.>"],
    "absences": ["<List of names of members who were absent for the vote.>"],
    "recusals": ["<List of names of members who recused themselves from voting.>"],
    "vote_result": "<MOTION PASSED | MOTION FAILED>",
    "appeal_result": "<If the agenda item involved an appeal, say whether the appeal was granted or denied; your options are: GRANTED | GRANTED_IN_PART | DENIED | N/A>",
    "project_result": "<What does the Commission's decision imply for the project proposed in the agenda item? For example, does it mean the project is approved, denied, sent back for revisions, etc.? Your options are: APPROVED | APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS | APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS | SENT BACK FOR REVISION | DELAYED | DENIED | APPLICATION WITHDRAWN>" 
}

Guidelines:
- Focus only on the agenda item provided; ignore other items in the minutes.
- Start by reading the agenda item and minutes fully to understand the deliberations and decisions made.
- Carefully compare the requested actions in the agenda item with the final decisions recorded in the minutes.
- If multiple votes were made on the item, use the final vote, but note the process of the deliberations in the summary and set multiple_votes to "YES".
- If a successful motion was to table the item, the project_result is "DELAYED".
- If the item was not voted and postponed to a future meeting, the project_result is "DELAYED".
- If the item was withdrawn by the applicant, the project_result is "APPLICATION WITHDRAWN".
- If there was an appeal but the item was not voted on or postponed, set appeal_result to "N/A".
- If there was no vote on the item, set mover, seconder, ayes, noes, abstentions, absences, recusals to empty lists or strings as appropriate, and set vote_result to "N/A".
- When listing names, use names as they appear in the minutes.
- Before producing the JSON, work through your reasoning inside <reasoning></reasoning> tags. In your reasoning:
    - Identify which parts of the minutes relate to the agenda item.
    - Note whether there were multiple motions or votes.
    - Determine the final motion and its outcome.
    - Work through what the decision means for the project.
- After your reasoning, return valid JSON only, no other text or markdown formatting.

AGENDA ITEM:

$agenda_item_text

MINUTES TEXT:

$minutes_text

""")

In [14]:
# write prompt to figures
with open(os.path.join(LOCAL_PATH, 'figures', 'agenda_split_prompt.tex'), 'w') as f:
    out = PROMPT.substitute(agenda_item_text="...", minutes_text="...")
    out = out.strip()
    out = out.replace('\n', '\\\\ \n')
    f.write(out)

In [ ]:
input_tokens = 0
output_tokens = 0
n_requests = 0
n_direct = 0
batch_num = 0
max_input_token_length = 0
max_input_token_date = ""
max_input_token_prompt = ""

rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)

t0 = time.time()
prompts_to_submit = []
for idx, row in meetings_df.iterrows():
    if idx < MIN_IDX or idx > MAX_IDX:
        continue

    date = row['date']
    year = date[0:4]

    print(f"{idx} {date} ...")

    # Input data
    agenda_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized.json")
    agenda_override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-override.json")
    minutes_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "minutes.txt")
    minutes_override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "minutes-manual-override.txt")
    if (not os.path.exists(agenda_filepath)) and (not os.path.exists(agenda_override_filepath)):
        print("No agenda text file found, skipping.")
        continue
    if (not os.path.exists(minutes_filepath)) and (not os.path.exists(minutes_override_filepath)):
        print("No minutes text file found, skipping.")
        continue
    if os.path.exists(agenda_override_filepath):
        with open(agenda_override_filepath, 'r', encoding='utf-8') as f:
            j = json.load(f)
    else:
        with open(agenda_filepath, 'r', encoding='utf-8') as f:
            j = json.load(f)
    if os.path.exists(minutes_override_filepath):
        with open(minutes_override_filepath, 'r', encoding='utf-8') as f:
            minutes_text = f.read()
    else:
        with open(minutes_filepath, 'r', encoding='utf-8') as f:
            minutes_text = f.read()

    # Output paths
    output_dir = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date)
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, "agenda-cleaned-summarized-minutes.json")
    if (not REPLACE_OUTPUT_FILE) and os.path.exists(output_filepath):
        print("Output file exists, skipping.")
        continue

    for k, v in enumerate(j):
        j[k]["ai_minutes_summary"] = {}

        agenda_item_title = j[k]["ai_summary"].get("title", "")
        if not is_casenum(agenda_item_title):
            continue
        agenda_item_text = j[k]["text"]
        prompt = PROMPT.substitute(agenda_item_text=agenda_item_text, minutes_text=minutes_text)

        # check if prompt is in cache
        if not REPLACE:
            prompt_hash = llt.get_hash(prompt)
            cached_response = llt.check_response_cache(prompt_hash, response_db_conn)
            if cached_response:
                if MODE=='WRITE':
                    response = cached_response[0]
                    logprob = cached_response[1]
                    perplexity = cached_response[2]
                    parsed = json.loads(response.split("</reasoning>")[-1].strip().removeprefix("```json").removesuffix("```").strip())
                    j[k]["ai_minutes_summary"] = parsed
                continue

        #my_tokens = llt.count_tokens(prompt, MODEL)
        input_tokens += len(prompt)/3.5
        output_tokens += output_tokens_per_request
        if len(prompt)/3.5 > max_input_token_length:
            max_input_token_length = len(prompt)/3.5
            max_input_token_date = date
            max_input_token_prompt = prompt
        n_requests += 1

        # add prompt to batch and check if batch is ready to submit
        if MODE=='BATCH':
            prompts_to_submit.append(prompt)
            if len(prompts_to_submit) >= REQUESTS_PER_BATCH:
                input_filename = f"summarize_agendas_batch_{batch_num}.jsonl"
                llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
                prompts_to_submit = []
                batch_num += 1
                print(f"    batch submitted")
        elif MODE=='WRITE':
            my_response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
            response = my_response['response']
            logprob = my_response['total_logprob']
            perplexity  = my_response['perplexity']
            n_direct += 1
            parsed = json.loads(response.split("</reasoning>")[-1].strip().removeprefix("```json").removesuffix("```").strip())
            j[k]["ai_minutes_summary"] = parsed
    
    if MODE=='WRITE':
        # write output file
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(j, f, indent=2, ensure_ascii=False)
        print(f"    wrote output file")

if (MODE=='BATCH') and (len(prompts_to_submit) > 0):
    input_filename = f"summarize_agendas_batch_{batch_num}.jsonl"
    llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    prompts_to_submit = []
    print(f"    batch submitted")

t1 = time.time()
print(f"    elapsed time: {(t1-t0)/60:.2f} minutes, {n_direct:,} direct requests")

batch_db_conn.close()
response_db_conn.close()

print(f"{n_direct:,} direct requests made")

0 2018-05-10 ...
    wrote output file
    elapsed time: 0.00 minutes, 0 direct requests
0 direct requests made


In [16]:
# Cost Estimate

total_input_cost = input_tokens / 1e6 * input_cost
total_output_cost = output_tokens / 1e6 * output_cost

print(f"Estimated number of requests: {n_requests:,.0f}")
print(f"Estimated total input tokens: {input_tokens:,.0f}")
print(f"Estimated total input cost: ${total_input_cost:.2f}")
print(f"Estimated total output tokens: {n_requests * output_tokens_per_request:,.0f}")
print(f"Estimated total output cost: ${total_output_cost:.2f}")
print(f"Estimated total cost: ${total_input_cost + total_output_cost:.2f}")
print(f"Max input token length: {max_input_token_length:,.0f} on date {max_input_token_date}")

Estimated number of requests: 0
Estimated total input tokens: 0
Estimated total input cost: $0.00
Estimated total output tokens: 0
Estimated total output cost: $0.00
Estimated total cost: $0.00
Max input token length: 0 on date 


In [19]:
j

[{'item_no': '1',
  'sub_item_no': '',
  'text': '       1.   DIRECTOR’S REPORT AND COMMISSION BUSINESS                               \n                                                                                    \n              •  Update on City Planning Commission Status Reports and Active Assignments\n                                                                                    \n                   Case No. CPC-2015-4703-VZC-ZV-SPR-ZAA-CU-CUB continued from March 8, 2018 to\n                   the City Planning Commission Meeting of May 10, 2018 will be on the Commission’s\n                   agenda of June 14, 2018.                                         \n              •  Legal actions and issues update                                    \n                                                                                    \n              •  Other Items of Interest                                            \n                                                           